In [ ]:
# pip install keras

In [10]:
import argparse
import csv
import os
import random
import time
from collections import deque
from datetime import datetime

import numpy as np

from tetris_env import TetrisEnv, extract_features
from keras_dqn_agent import DQNAgent

2026-04-21 11:29:33.511946: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-21 11:29:33.548861: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-21 11:29:38.373933: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-21 11:29:46.926018: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [11]:
"""
Training Script 
"""

# Enviorment
ROWS    = 14
COLS    = 8
RENDER  = False

# Training
EPISODES        = 10_000
WARMUP_EPISODES = 100

# Agent
GAMMA           = 0.99
LR              = 1e-3          
EPSILON_START   = 1.0
EPSILON_MIN     = 0.05
EPSILON_DECAY   = 0.9995

# Replay Buffer
BATCH_SIZE      = 512           
BUFFER_SIZE     = 50_000

# Target Network
TAU             = 0.01

# Agent
HIDDEN_UNITS    = [128, 64, 32]

# Logging and Saving
SAVE_EVERY      = 250
LOG_EVERY       = 10
WEIGHTS_PATH    = "tetris_dqn.weights.h5"
LOG_CSV         = "training_log.csv"

# curriculum
CURRICULUM_START     = 0
CURRICULUM_THRESHOLD = 50
CURRICULUM_WINDOW    = 100


# CSV helpers
CSV_FIELDS = [
    "episode", "timestamp", "total_reward", "pieces_placed", "lines_cleared",
    "avg_reward_100", "avg_pieces_100", "epsilon", "avg_loss_200", "curriculum_level",
]

def init_csv(path):
    if not os.path.exists(path):
        with open(path, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

def log_csv(path, row):
    with open(path, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row)


# Helpers
def make_env(render=False) -> TetrisEnv:
    env = TetrisEnv(rows=ROWS, cols=COLS, render=render)
    env._curriculum_level = CURRICULUM_START
    return env

def get_obs(time_step) -> np.ndarray:
    return np.asarray(time_step.observation, dtype=np.float32)


# Training loop
def train(load_weights: bool = False, log_csv_path: str = LOG_CSV):
    env          = make_env(render=RENDER)
    feature_size = env.feature_size

    agent = DQNAgent(
        feature_size  = feature_size,
        gamma         = GAMMA,
        lr            = LR,
        epsilon_start = EPSILON_START,
        epsilon_min   = EPSILON_MIN,
        epsilon_decay = EPSILON_DECAY,
        batch_size    = BATCH_SIZE,
        buffer_size   = BUFFER_SIZE,
        tau           = TAU,
        hidden_units  = HIDDEN_UNITS,
    )

    if load_weights:
        try:
            agent.load(WEIGHTS_PATH)
            agent.epsilon = agent.epsilon_min
        except Exception as e:
            print(f"[train] Could not load weights: {e}. Starting fresh.")

    init_csv(log_csv_path)

    rewards_win = deque(maxlen=100)
    pieces_win  = deque(maxlen=CURRICULUM_WINDOW)
    losses_win  = deque(maxlen=200)
    best_avg    = -float("inf")

    print(f"\n{'─'*68}")
    print(f"  Tetris Afterstate Agent  |  board {ROWS}×{COLS}")
    print(f"  Features: {feature_size}  |  Hidden: {HIDDEN_UNITS}")
    print(f"  LR={LR}  batch={BATCH_SIZE}  ε_decay={EPSILON_DECAY}")
    print(f"  Logging → {log_csv_path}")
    print(f"{'─'*68}\n")

    for episode in range(1, EPISODES + 1):
        ts           = env._reset()
        state_feats  = get_obs(ts)

        total_reward  = 0.0
        pieces_placed = 0

        while True:
            # Get all valid placements and their resulting board features
            next_states = env.get_next_states()

            # no valid moves
            if not next_states:
                break   

            # Choose action
            if episode <= WARMUP_EPISODES:
                action = random.choice(list(next_states.keys()))
            else:
                action = agent.act(next_states)

            chosen_features, lines = next_states[action]

            ts = env._step(action)
            reward = float(ts.reward)
            done = ts.is_last()
            pieces_placed = env._game.pieces_placed
            total_reward += reward

            # Get best value state
            if done:
                # use a zero feature vector as the dummy next state
                next_best_features = np.zeros(feature_size, dtype=np.float32)
            else:

                next_next_states = env.get_next_states()

                if next_next_states:

                    # Pick the features of the best next placement
                    next_actions = list(next_next_states.keys())
                    next_feats = np.array([next_next_states[a][0] for a in next_actions], dtype=np.float32)
                    next_vals = agent.online_net(next_feats, training=False).numpy().flatten()
                    next_best_features = next_feats[int(np.argmax(next_vals))]

                else:
                    next_best_features = np.zeros(feature_size, dtype=np.float32)

            # Add to replay buffer
            agent.remember(chosen_features, reward, next_best_features, float(done))
            state_feats = chosen_features

            if episode > WARMUP_EPISODES:
                loss = agent.learn()

                if loss is not None:
                    losses_win.append(loss)

            if done:
                break

        rewards_win.append(total_reward)
        pieces_win.append(pieces_placed)

        # Logging
        avg_reward = float(np.mean(rewards_win))
        avg_pieces = float(np.mean(pieces_win))
        avg_loss   = float(np.mean(losses_win)) if losses_win else float("nan")

        log_csv(log_csv_path, {
            "episode":          episode,
            "timestamp":        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "total_reward":     round(total_reward, 3),
            "pieces_placed":    pieces_placed,
            "lines_cleared":    env._game.score,
            "avg_reward_100":   round(avg_reward, 3),
            "avg_pieces_100":   round(avg_pieces, 2),
            "epsilon":          round(agent.epsilon, 5),
            "avg_loss_200":     round(avg_loss, 5) if not np.isnan(avg_loss) else "",
            "curriculum_level": env._curriculum_level,
        })

        # Curriculum
        if (
            len(pieces_win) == CURRICULUM_WINDOW
            and avg_pieces > CURRICULUM_THRESHOLD
            and env._curriculum_level > 0
        ):
            env.reduce_curriculum()
            print(f"[Curriculum Reduced] Level: {env._curriculum_level} (avg pieces={avg_pieces:.1f})")
            pieces_win.clear()

        if episode % LOG_EVERY == 0:
            print(
                f"Ep {episode:>6} | "
                f"R {total_reward:>7.1f} | "
                f"AvgR {avg_reward:>7.1f} | "
                f"Pcs {pieces_placed:>4} | "
                f"AvgPcs {avg_pieces:>5.1f} | "
                f"Lines {env._game.score:>3} | "
                f"ε {agent.epsilon:.4f} | "
                f"Loss {avg_loss:.4f}"
            )

        if episode % SAVE_EVERY == 0:
            agent.save(WEIGHTS_PATH)
            if avg_reward > best_avg:
                best_avg = avg_reward
                agent.save("tetris_dqn_best.weights.h5")
                print(f" New best avg reward: {best_avg:.2f}  (ep {episode})")

    print("\nTraining complete.")
    agent.save(WEIGHTS_PATH)


#  Demo
def demo(episodes: int = 5):
    env = make_env(render=True)
    env._curriculum_level = 0

    agent = DQNAgent(feature_size=env.feature_size, hidden_units=HIDDEN_UNITS)
    agent.load(WEIGHTS_PATH)

    for ep in range(1, episodes + 1):
        env._reset()
        total_reward = 0.0

        while True:
            next_states = env.get_next_states()
            if not next_states:
                break
            action       = agent.act_greedy(next_states)
            ts           = env._step(action)
            total_reward += float(ts.reward)
            time.sleep(0.3)
            if ts.is_last():
                break

        print(f"Demo ep {ep}: reward={total_reward:.1f}  pieces={env._game.pieces_placed}  lines={env._game.score}")

In [ ]:
""" Graph Creating """

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(LOG_CSV)

# Average Loss
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["episode"], df["avg_loss_200"])
ax.set_title("Average Loss (200 episodes)")
ax.set_xlabel("Episode")
ax.set_ylabel("Loss")
plt.tight_layout()
plt.savefig("Graphs/loss.png")
plt.show()

# Total reward
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["episode"], df["total_reward"], color="green")
ax.set_title("Reward")
ax.set_xlabel("Episode")
ax.set_ylabel("Reward")
plt.tight_layout()
plt.savefig("Graphs/reward.png")
plt.show()

# Eval reward
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["episode"], df["avg_reward_100"], color="green")
ax.set_title("Avg Reward (100 Episodes)")
ax.set_xlabel("Episode")
ax.set_ylabel("Reward")
plt.tight_layout()
plt.savefig("Graphs/avg_reward.png")
plt.show()

# Pieces placed
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["episode"], df["pieces_placed"], color="red")
ax.set_title("Pieces Placed")
ax.set_xlabel("Episode")
ax.set_ylabel("Pieces Placed")
plt.tight_layout()
plt.savefig("Graphs/placed.png")
plt.show()

# Pieces placed
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["episode"], df["avg_pieces_100"], color="red")
ax.set_title("Avg Pieces Placed (100 episodes)")
ax.set_xlabel("Episode")
ax.set_ylabel("Pieces Placed")
plt.tight_layout()
plt.savefig("Graphs/avg_placed.png")
plt.show()

# Lines cleared
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["episode"], df["lines_cleared"], color="blue")
ax.set_title("Lines Cleared")
ax.set_xlabel("Episode")
ax.set_ylabel("Lines")
plt.tight_layout()
plt.savefig("Graphs/cleared.png")
plt.show()

# Epsilon decay
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["episode"], df["epsilon"], color="orange")
ax.set_title("Epsilon (Exploration)")
ax.set_xlabel("Episode")
ax.set_ylabel("ε")
plt.tight_layout()
plt.savefig("Graphs/exploration.png")
plt.show()

In [ ]:
train(True)

In [12]:
demo(1)

2026-04-21 11:29:50.078699: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


[Agent] Loaded: tetris_dqn.weights.h5
[0. 0. 2. 2. 2. 2. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[1. 1. 0. 0. 0. 0. 0. 0.]
[0. 1. 1. 0. 0. 0. 0. 0.]

[0. 0. 0. 2. 2. 0. 0. 0.]
[0. 0. 2. 2. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[1. 1. 0. 0. 0. 0. 0. 0.]
[0. 1. 1. 1. 1. 1. 1. 0.]

[0. 0. 0. 2. 0. 0. 0. 0.]
[0. 0. 0. 2. 0. 0. 0. 0.]
[0. 0. 2. 2. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0.